In [ ]:
import hashlib
import math
from pathlib import Path

import torch
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

torch.manual_seed(2026)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CONTEXT = 64
path = Path("transcript.txt")
if not path.exists():
    raise FileNotFoundError("Place BeRP transcript.txt beside this notebook")

records = []
for line in path.read_text(encoding="utf-8", errors="replace").splitlines():
    parts = line.strip().split(maxsplit=1)
    if len(parts) == 2 and parts[1].strip():
        records.append((parts[0], parts[1].lower().split()))

def bucket(record_id):
    return int.from_bytes(hashlib.sha256(record_id.encode()).digest()[:8], "big") % 10

train = [words for uid, words in records if bucket(uid) < 8]
val = [(uid, words) for uid, words in records if bucket(uid) == 8]
val.sort(key=lambda row: hashlib.sha256(row[0].encode()).digest())
val = [words for _, words in val]
assert train and val, "Expected nonempty training and validation splits"
# Bucket 9 is reserved for the instructor's final evaluation.

vocab = ["<pad>", "<bos>", "<eos>", "<unk>"]
vocab += sorted(set(w for sent in train for w in sent) - set(vocab))
stoi = {w: i for i, w in enumerate(vocab)}
PAD, BOS, EOS, UNK = (stoi[w] for w in vocab[:4])
VOCAB_SIZE = len(vocab)

class LMData(Dataset):
    def __init__(self, sentences):
        self.pairs = []
        for words in sentences:
            ids = [BOS] + [stoi.get(w, UNK) for w in words] + [EOS]
            for start in range(0, len(ids) - 1, CONTEXT):
                self.pairs.append((
                    torch.tensor(ids[start:start + CONTEXT]),
                    torch.tensor(ids[start + 1:start + CONTEXT + 1]),
                ))

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, i):
        return self.pairs[i]

def collate(batch):
    return tuple(pad_sequence(parts, batch_first=True, padding_value=PAD)
                 for parts in zip(*batch))

train_loader = DataLoader(LMData(train), batch_size=64, shuffle=True, collate_fn=collate)
quick_loader = DataLoader(LMData(val[:256]), batch_size=64, collate_fn=collate)
val_loader = DataLoader(LMData(val), batch_size=64, collate_fn=collate)
print(f"device={DEVICE}, vocab={VOCAB_SIZE}, train={len(train)}, val={len(val)}")

In [ ]:
def score(model, loader):
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    if not 0 < params <= 2_000_000:
        raise ValueError(f"Parameter budget exceeded: {params:,} (limit 2,000,000)")
    device = next(model.parameters()).device
    was_training = model.training
    model.eval()
    nll, tokens = 0.0, 0
    try:
        with torch.inference_mode():
            for x, y in loader:
                x, y = x.to(device), y.to(device)
                logits = model(x)
                if logits.shape != (x.size(0), x.size(1), VOCAB_SIZE):
                    raise ValueError("Expected logits (batch, time, VOCAB_SIZE)")
                nll += F.cross_entropy(
                    logits.reshape(-1, VOCAB_SIZE), y.reshape(-1),
                    ignore_index=PAD, reduction="sum",
                ).item()
                tokens += (y != PAD).sum().item()
    finally:
        model.train(was_training)
    ce = nll / tokens
    ppl = math.exp(ce)
    points = min(30.0, 600.0 / (ppl * (max(params, 100_000) / 1_000_000) ** 0.15))
    print(f"CE={ce:.3f} | PPL={ppl:.2f} | params={params:,} | score={points:.2f}/30")
    return {"ce": ce, "ppl": ppl, "params": params, "points": points}